# Phase 09 — Moteur de Décision MILP (Optimisation Couplée)

> **⚙️ Résultat de cette phase :** Le code d'optimisation est implémenté de manière modulaire dans `src/optimization/milp_engine.py` pour être utilisé par le simulateur.

## 🎯 Objectif
Ce notebook implémente le cœur algorithmique du système SON (Self-Organizing Network). L'objectif est de résoudre le problème de congestion réseau non pas de manière isolée antenne par antenne, mais de manière **couplée et globale**.

## 🧠 Logique Mathématique (MILP)
Nous utilisons la **Programmation Linéaire en Nombres Entiers (MILP)** pour choisir, pour chaque antenne, l'un des 7 niveaux d'Offset A3 disponibles.

### 1. Variables de décision
- $z_{a,k} \in \{0, 1\}$ : Vaut 1 si l'antenne $a$ applique le niveau de délestage $k$.
- $u_a \ge 0$ : Volume de trafic non satisfait (excès résiduel) sur l'antenne $a$.

### 2. Contraintes
- **Unicité (C1)** : $\sum_k z_{a,k} = 1$. Chaque antenne ne peut avoir qu'une seule configuration à la fois.
- **Conservation et Couplage (C2/C3)** : Le volume final sur une antenne $b$ est :
  $$V_{final, b} = V_{initial, b} - \text{Délestage}_b + \sum_{\text{voisins } a} \text{Reçu}_{a \to b}$$
  Cette contrainte est cruciale : elle garantit qu'en délestant une antenne $a$, on ne crée pas une congestion pire chez sa voisine $b$.

### 3. Objectif
$$\min \sum_a u_a$$
On minimise l'excès de trafic global sur l'ensemble du cluster de 600 cellules.

In [1]:
import numpy as np
import polars as pl
import sys
from pathlib import Path
from pyomo.environ import *

# Import du module restauré
sys.path.append('../scripts')
from online.milp_builder import build_H_matrices, solve_congestion_milp

print("Moteur MILP chargé avec logique de couplage global.")

Moteur MILP chargé avec logique de couplage global.


## 🛠️ Justification du levier A3 Offset
Comme précisé dans le guide, l'A3 Offset est le seul levier SON standardisé (3GPP) permettant un pilotage dynamique sans risque d'instabilité majeure (ping-pongs). En modifiant virtuellement la frontière de couverture, on déplace physiquement les utilisateurs vers les antennes les moins chargées.